In [ ]:
%pip install pandas matplotlib seaborn openpyxl -q

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
arquivo = "Tabela_13_Meio_de_transporte.xlsx"
df = pd.read_excel(arquivo, sheet_name="Exemplo gráfico Brasil", skiprows=1)
df.columns = ["Meio de Transporte", "Branca", "Preta ou Parda", "Indígena"]
df = df.dropna(subset=["Meio de Transporte"])
for col in ["Branca", "Preta ou Parda", "Indígena"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
cores = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0', '#ffb3e6']
grupos = ['Branca', 'Preta ou Parda', 'Indígena']

for i, col in enumerate(grupos):
    axes[i].pie(
        df[col],
        labels=df['Meio de Transporte'],
        autopct='%1.3f%%',
        startangle=180,
        colors=cores,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.2},
        textprops={'fontsize': 8},
    )
    axes[i].set_title(f'Cor/Raça: {col}', fontsize=12, fontweight='bold')

plt.suptitle('Meios de transporte utilizados por mulheres, segundo cor ou raça — Brasil (2022)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('grafico_transporte.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
comparacao = df.set_index("Meio de Transporte")
comparacao["Branca - Preta ou Parda (p.p.)"] = comparacao["Branca"] - comparacao["Preta ou Parda"]
comparacao["Branca - Indígena (p.p.)"] = comparacao["Branca"] - comparacao["Indígena"]
comparacao["Preta ou Parda - Indígena (p.p.)"] = comparacao["Preta ou Parda"] - comparacao["Indígena"]
display(comparacao.round(2))

### Análise dos gráficos

Os valores representam a **distribuição percentual dentro de cada grupo de cor ou raça**, e as diferenças abaixo são expressas em **pontos percentuais (p.p.)**, não em variação percentual relativa.

| Meio de transporte | Brancas | Pretas ou pardas | Indígenas | Diferença: brancas − pretas/pardas | Diferença: brancas − indígenas |
|---|---:|---:|---:|---:|---:|
| A pé | 19,4% | 24,8% | 37,5% | −5,4 p.p. | −18,1 p.p. |
| Bicicleta | 3,2% | 5,2% | 5,9% | −2,0 p.p. | −2,7 p.p. |
| Motocicleta ou mototáxi | 9,9% | 14,2% | 15,4% | −4,3 p.p. | −5,5 p.p. |
| Automóvel, táxi ou assemelhados | 41,8% | 20,6% | 15,2% | +21,2 p.p. | +26,6 p.p. |
| Transporte coletivo | 25,2% | 34,6% | 22,4% | −9,4 p.p. | +2,8 p.p. |
| Outros | 0,5% | 0,6% | 3,6% | −0,1 p.p. | −3,1 p.p. |

As mulheres brancas apresentam maior participação no uso de automóvel (41,8%), 21,2 p.p. acima das mulheres pretas ou pardas. Entre as mulheres pretas ou pardas, o transporte coletivo tem maior participação (34,6%), 9,4 p.p. acima das brancas e 12,2 p.p. acima das indígenas. Entre as mulheres indígenas, caminhar é o meio mais frequente (37,5%), 18,1 p.p. acima das brancas e 12,7 p.p. acima das pretas ou pardas.

Essas diferenças descrevem a distribuição observada, mas não demonstram suas causas. A tabela contém percentuais, e não quantidades absolutas de mulheres.

In [ ]:
# Leitura da aba completa e organização das colunas por cor/raça e transporte
bruto = pd.read_excel(arquivo, sheet_name="BR GR UF MU", header=None, skiprows=1)
modos = ["A pé", "Bicicleta", "Motocicleta ou Mototaxi",
         "Automóvel, taxi ou assemelhados", "Transporte coletivo", "Outros"]
colunas = ["Localidade"] + [f"{raca} | {modo}" for raca in
                           ["Branca", "Preta ou Parda", "Indígena"] for modo in modos]
dados = bruto.iloc[2:, :19].copy()
dados.columns = colunas
dados = dados.dropna(subset=["Localidade"]).reset_index(drop=True)
for coluna in colunas[1:]:
    dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce")
dados.head()

In [ ]:
# As primeiras 27 unidades da federação aparecem depois de Brasil e das 5 regiões
estados = dados.iloc[6:33].copy()
display(estados[["Localidade", "Branca | Transporte coletivo",
                 "Preta ou Parda | Transporte coletivo",
                 "Indígena | Transporte coletivo"]])

In [ ]:
# "Em geral": média simples dos percentuais disponíveis das três categorias de cor/raça.
# Não é a proporção total de mulheres: para isso seriam necessários os totais populacionais por grupo.
col_coletivo = [f"{raca} | Transporte coletivo" for raca in
                ["Branca", "Preta ou Parda", "Indígena"]]
estados["Média transporte coletivo (%)"] = estados[col_coletivo].mean(axis=1)
ranking_estados = estados[["Localidade", "Média transporte coletivo (%)"]].dropna().sort_values(
    "Média transporte coletivo (%)", ascending=False
)
print("Estado com maior média de transporte coletivo:")
display(ranking_estados.head(1))
print("Estado com menor média de transporte coletivo:")
display(ranking_estados.tail(1))

In [ ]:
# Municípios: valores percentuais por grupo, não números absolutos de mulheres.
municipios = dados.iloc[33:].copy()
col_bicicleta = [f"{raca} | Bicicleta" for raca in
                 ["Branca", "Preta ou Parda", "Indígena"]]
col_carro = [f"{raca} | Automóvel, taxi ou assemelhados" for raca in
             ["Branca", "Preta ou Parda", "Indígena"]]
municipios["Média bicicleta (%)"] = municipios[col_bicicleta].mean(axis=1)
municipios["Média carro (%)"] = municipios[col_carro].mean(axis=1)

print("Município com maior média percentual de uso de bicicleta:")
display(municipios[["Localidade", "Média bicicleta (%)"]].dropna()
        .sort_values("Média bicicleta (%)", ascending=False).head(1))
print("Município com maior média percentual de uso de carro:")
display(municipios[["Localidade", "Média carro (%)"]].dropna()
        .sort_values("Média carro (%)", ascending=False).head(1))

In [ ]:
# Rankings completos para conferência
display(ranking_estados)
display(municipios[["Localidade", "Média bicicleta (%)", "Média carro (%)"]]
        .sort_values("Média bicicleta (%)", ascending=False).head(10))
display(municipios[["Localidade", "Média bicicleta (%)", "Média carro (%)"]]
        .sort_values("Média carro (%)", ascending=False).head(10))